# 02 - From behavioral trials to the first neural RDM

This notebook completes the first analysis milestone: align every **vision** and **imagery** trial to the correct beta, average repeated target patterns, and compute neural representational dissimilarity matrices (RDMs).

The output is a pilot neural result, not yet the project's feature-level hypothesis test. We deliberately exclude attention because it has two beta epochs per behavioral trial.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'nsdimagery').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'nsdimagery').is_dir():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks/ directory.')
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

from nsdimagery.io import (
    build_event_table, extract_masked_betas, find_data_root,
    load_roi, load_target_table, paths_for_subject, summarize_glmsingle_design,
)

sns.set_theme(style='whitegrid', context='notebook')
DATA_ROOT = find_data_root(REPO_ROOT)
print('Repository:', REPO_ROOT)
print('Data root:', DATA_ROOT)

## 1. The alignment rule

A beta index and a behavioral row are one-to-one inside vision and imagery runs. Their target labels come from different behavioral columns:

| Task | Target label | Reason |
|---|---|---|
| Vision | `CONDITION` | This is the image actually shown. `CUE` can differ during mismatch trials. |
| Imagery | `CUE` | This tells the participant which target to imagine. |

The beta file follows acquisition order. Attention occupies beta blocks too, so we preserve those gaps when assigning the zero-based `beta_index`.

In [ ]:
SUBJECT = 1
ROI_NAME = 'nsdgeneral'
MAX_VOXELS = 2000  # Fast pilot. Use None only after this notebook works end to end.
RANDOM_SEED = 2026

targets = load_target_table(DATA_ROOT)
display(targets)
assert len(targets) == 18
assert targets.groupby('stimulus_set').size().eq(6).all()

In [ ]:
image_targets = targets[targets['image_path'].notna()]
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, row in zip(axes.flat, image_targets.itertuples(index=False)):
    ax.imshow(Image.open(row.image_path))
    ax.set_title(f'set {row.stimulus_set}: {row.target_code}\n{row.target_name}', fontsize=8)
    ax.axis('off')
plt.tight_layout()

Set A contains six controlled geometric targets. Set B contains six fixed natural images. Set C contains six concepts rather than six fixed image files, so its later feature analysis needs separate treatment.

In [ ]:
design_summary = summarize_glmsingle_design(DATA_ROOT)
display(design_summary)
assert design_summary['ok'].all()
assert design_summary['n_modeled_events'].sum() == 720

events = build_event_table(DATA_ROOT, SUBJECT)
core_columns = [
    'subject', 'run_name', 'task', 'stimulus_set', 'TRIAL',
    'beta_index', 'beta_number', 'target_code', 'target_name', 'repeat',
]
display(events[core_columns].head(12))
print('Vision + imagery rows:', len(events))
print('Unique beta indices:', events['beta_index'].nunique())
print('Smallest/largest beta index:', events['beta_index'].min(), events['beta_index'].max())
assert len(events) == 432
assert events['beta_index'].is_unique
assert events['beta_index'].between(0, 719).all()

In [ ]:
run_checks = (
    events.groupby(['run_name', 'task', 'stimulus_set'])
    .agg(n_trials=('TRIAL', 'size'), first_beta=('beta_number', 'min'), last_beta=('beta_number', 'max'))
    .reset_index()
)
display(run_checks)

repeat_checks = (
    events.groupby(['stimulus_set', 'target_code', 'task']).size()
    .unstack('task')
    .sort_index()
)
display(repeat_checks)
assert repeat_checks['vision'].eq(8).all()
assert repeat_checks['imagery'].eq(16).all()

vision = events[events['task'] == 'vision']
print('Vision trials where the cue differs from the shown target:',
      (vision['CUE'].astype(str) != vision['CONDITION'].astype(str)).sum())

### Read this table before proceeding

- `TRIAL` is one-based within a run.
- `beta_index` is zero-based for Python indexing into the 720 beta events.
- `beta_number` is the same event numbered 1 through 720 for human-readable reports.
- `repeat` counts repetitions of a target within task and set.
- Missing beta numbers are expected here: they belong to the excluded attention runs.

The mismatch count in vision is also expected. Those trials show one target while asking about another cue, which is why vision must be labeled from `CONDITION`.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
for task, frame in events.groupby('task'):
    ax.scatter(frame['beta_number'], np.full(len(frame), task), s=8, alpha=.7, label=task)
ax.set(xlabel='beta number in the 720-event file', ylabel='', title='Selected beta blocks')
ax.legend(title='task')
plt.tight_layout()

## 2. Extract trial patterns from one ROI

We begin with a reproducible sample of `nsdgeneral` voxels. Two thousand voxels are enough to test alignment and generate a pilot RDM without making the notebook sluggish. The final analysis should repeat this across all subjects and use the planned ROI definitions.

In [ ]:
subject_paths = paths_for_subject(DATA_ROOT, SUBJECT)
roi_array, roi_image = load_roi(DATA_ROOT, SUBJECT, ROI_NAME)
all_betas, voxel_coordinates = extract_masked_betas(
    subject_paths['beta'], roi_array, max_voxels=MAX_VOXELS, seed=RANDOM_SEED
)
trial_patterns = all_betas[events['beta_index'].to_numpy()]
print('All beta events x selected voxels:', all_betas.shape)
print('Selected trial patterns:', trial_patterns.shape)
print('ROI voxels before sampling:', int((roi_array > 0).sum()))
assert trial_patterns.shape == (432, len(voxel_coordinates))

## 3. Normalize within run

Scanner and run-level offsets can dominate distances. For this pilot, each voxel is centered and scaled across the 48 selected trials inside its run. We then average repeated trials. The raw percent-signal-change values remain available in `trial_patterns`; the RDM uses `normalized_patterns`.

In [ ]:
normalized_patterns = np.empty_like(trial_patterns, dtype=np.float32)
normalization_qc = []
for run_name, positions in events.groupby('run_name', sort=False).indices.items():
    positions = np.asarray(positions)
    block = trial_patterns[positions]
    center = block.mean(axis=0, keepdims=True)
    scale = block.std(axis=0, ddof=1, keepdims=True)
    scale[scale == 0] = 1
    normalized_patterns[positions] = (block - center) / scale
    normalization_qc.append({
        'run_name': run_name,
        'raw_mean': float(block.mean()),
        'raw_sd': float(block.std()),
        'normalized_mean': float(normalized_patterns[positions].mean()),
        'normalized_sd': float(normalized_patterns[positions].std()),
    })
normalization_qc = pd.DataFrame(normalization_qc)
display(normalization_qc)

## 4. Average repetitions into 36 target patterns

For each task there are three sets and six targets: $2 \times 3 \times 6 = 36$ averaged patterns. Vision averages 8 trials per target; imagery averages 16.

In [ ]:
group_columns = ['task', 'stimulus_set', 'target_number', 'target_code', 'target_name']
pattern_rows = []
averaged_patterns = []
for key, positions in events.groupby(group_columns, sort=True).indices.items():
    positions = np.asarray(positions)
    pattern_rows.append(dict(zip(group_columns, key), n_trials=len(positions)))
    averaged_patterns.append(normalized_patterns[positions].mean(axis=0))

pattern_table = pd.DataFrame(pattern_rows)
averaged_patterns = np.stack(averaged_patterns)
display(pattern_table)
print('Averaged target patterns x voxels:', averaged_patterns.shape)
assert averaged_patterns.shape[0] == 36

## 5. Compute and display neural RDMs

Each cell is correlation distance, $1-r$, between two multivoxel target patterns. A small value means the ROI expresses similar patterns for the two targets; a large value means the patterns differ. The diagonal is zero by definition.

In [ ]:
def correlation_rdm(patterns):
    rdm = squareform(pdist(patterns, metric='correlation'))
    np.fill_diagonal(rdm, 0)
    return rdm

def upper_triangle(rdm):
    return rdm[np.triu_indices_from(rdm, k=1)]

neural_rdms = {}
rdm_labels = {}
for stimulus_set in ['A', 'B', 'C']:
    for task in ['vision', 'imagery']:
        select = (
            pattern_table['stimulus_set'].eq(stimulus_set)
            & pattern_table['task'].eq(task)
        )
        rows = pattern_table.loc[select].sort_values('target_number')
        indices = rows.index.to_numpy()
        neural_rdms[(stimulus_set, task)] = correlation_rdm(averaged_patterns[indices])
        rdm_labels[(stimulus_set, task)] = rows['target_name'].tolist()

rdm_vmax = max(float(rdm.max()) for rdm in neural_rdms.values())
fig, axes = plt.subplots(3, 2, figsize=(13, 16))
for row_index, stimulus_set in enumerate(['A', 'B', 'C']):
    for column_index, task in enumerate(['vision', 'imagery']):
        ax = axes[row_index, column_index]
        key = (stimulus_set, task)
        labels = rdm_labels[key]
        sns.heatmap(
            neural_rdms[key], square=True, cmap='viridis', vmin=0, vmax=rdm_vmax, ax=ax,
            xticklabels=labels, yticklabels=labels, cbar_kws={'label': '1 - correlation'},
        )
        ax.set_title(f'Set {stimulus_set} - {task}')
        ax.tick_params(axis='x', rotation=60, labelsize=8)
        ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()

## 6. Two sanity checks before feature RSA

First, compare odd and even repetitions. This is a rough split-half reliability estimate: do repeated measurements recover similar target geometry? Second, correlate the vision and imagery RDMs. Both correlations use only 15 off-diagonal distances, so treat them as diagnostics rather than inferential results.

In [ ]:
def rdm_from_event_subset(frame):
    rows = []
    for target_number, event_indices in frame.groupby('target_number').groups.items():
        rows.append((target_number, normalized_patterns[np.asarray(event_indices)].mean(axis=0)))
    rows.sort(key=lambda item: item[0])
    return correlation_rdm(np.stack([pattern for _, pattern in rows]))

diagnostics = []
for stimulus_set in ['A', 'B', 'C']:
    for task in ['vision', 'imagery']:
        frame = events[
            events['stimulus_set'].eq(stimulus_set) & events['task'].eq(task)
        ]
        odd_rdm = rdm_from_event_subset(frame[frame['repeat'] % 2 == 1])
        even_rdm = rdm_from_event_subset(frame[frame['repeat'] % 2 == 0])
        reliability = spearmanr(upper_triangle(odd_rdm), upper_triangle(even_rdm)).statistic
        diagnostics.append({
            'stimulus_set': stimulus_set, 'task': task,
            'split_half_rho': reliability,
        })
diagnostics = pd.DataFrame(diagnostics)

cross_condition = []
for stimulus_set in ['A', 'B', 'C']:
    rho = spearmanr(
        upper_triangle(neural_rdms[(stimulus_set, 'vision')]),
        upper_triangle(neural_rdms[(stimulus_set, 'imagery')]),
    ).statistic
    cross_condition.append({'stimulus_set': stimulus_set, 'vision_imagery_rho': rho})
cross_condition = pd.DataFrame(cross_condition)

display(diagnostics)
display(cross_condition)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.barplot(data=diagnostics, x='stimulus_set', y='split_half_rho', hue='task', ax=axes[0])
axes[0].set(title='Split-half neural RDM reliability', ylabel='Spearman rho', ylim=(-1, 1))
sns.barplot(data=cross_condition, x='stimulus_set', y='vision_imagery_rho', color='0.35', ax=axes[1])
axes[1].set(title='Vision-imagery neural RDM similarity', ylabel='Spearman rho', ylim=(-1, 1))
for ax in axes:
    ax.axhline(0, color='black', linewidth=.8)
plt.tight_layout()

### How to interpret this checkpoint

- Positive split-half reliability means the target geometry is reproducible enough to compare with feature RDMs.
- Weak reliability is a QC warning, not evidence that imagery lacks structure. Increase the voxel sample, inspect ROI choice, and aggregate all subjects before interpreting it biologically.
- Positive vision-imagery similarity suggests some target geometry transfers across conditions. It still does not tell us whether that geometry is low-level or semantic.
- Set C is useful conceptually, but its vision trials do not correspond to one fixed image per concept. Keep it exploratory until its exact rendered frames are modeled.

## 7. Save the verified intermediate products

These small files let the next notebook start from an audited event table and neural RDMs. They do not duplicate the large beta file.

In [ ]:
OUTPUT_DIR = REPO_ROOT / 'outputs' / '02_neural_rdm'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
subject_label = f'subj{SUBJECT:02d}'
events.to_csv(OUTPUT_DIR / f'{subject_label}_event_table.csv', index=False)
diagnostics.to_csv(OUTPUT_DIR / f'{subject_label}_{ROI_NAME}_diagnostics.csv', index=False)
cross_condition.to_csv(OUTPUT_DIR / f'{subject_label}_{ROI_NAME}_cross_condition.csv', index=False)
np.savez_compressed(
    OUTPUT_DIR / f'{subject_label}_{ROI_NAME}_neural_rdms.npz',
    **{f'set_{stimulus_set}_{task}': rdm for (stimulus_set, task), rdm in neural_rdms.items()},
)
print('Saved to:', OUTPUT_DIR)

## 8. Where this leaves the project

After this notebook runs successfully, **Day 1 of the MVP is complete**: the trial-to-beta mapping is verified and the first neural RDM exists.

The shortest route to the first feature-level result is now:

1. Extract low-level image features (HOG) and high-level semantic features (CLIP) for sets A and B.
2. Build their feature RDMs and correlate them with each subject's vision and imagery neural RDMs.
3. Repeat across all eight subjects and summarize subject-level RSA values.
4. Split early visual and higher visual ROIs, then test the planned condition-by-feature-level-by-region contrast with permutation/bootstrap uncertainty.

Notebook 03 should implement steps 1-3 for the first `nsdgeneral` result. Notebook 04 should add the anatomical split and inference needed for the main defensible figure.

This notebook uses the existing `nsdimagery` conda environment. Before notebook 03, we will extend `environment.yml` with HOG and CLIP dependencies rather than installing packages ad hoc.

References: [NSD Data Manual](https://cvnlab.slite.page/p/CT9Fwl4_hc/NSD-Data-Manual) and [NSD-Imagery paper](https://arxiv.org/abs/2506.06898).